In [4]:
!nvidia-smi

Tue May 12 08:02:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.3/495.3 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 17.0 MB/s eta 0:00:00


In [6]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 33.2 MB/s eta 0:00:00


In [5]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00


In [1]:
import sys
print(sys.executable)

/usr/bin/python3


In [1]:
import numpy
import torch
import evaluate
print(numpy.__version__)
print(torch.__version__)
print(evaluate.__version__)

2.0.2
2.10.0+cu128
0.4.6


In [3]:
from transformers import pipeline, set_seed, AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import evaluate
import matplotlib.pyplot as plt
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

#model_ckpt = "sshleifer/distilbart-cnn-12-6"
model_ckpt = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully


In [5]:
# !wget
# !unzip summarizer-data.zip
import urllib.request
import zipfile

url = "https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip"
zip_path = "summarizer-data.zip"

# download
urllib.request.urlretrieve(url, zip_path)

print("Download completed")

# unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(".")

print("Extraction completed")

Download completed
Extraction completed


In [6]:
dataset_samsum = load_from_disk('samsum_dataset')
dataset_samsum

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})

In [7]:
def convert_examples_to_features(example_batch):

    input_encodings = tokenizer(
        example_batch['dialogue'],
        max_length=1024,
        truncation=True
    )

    target_encodings = tokenizer(
        text_target=example_batch['summary'],
        max_length=128,
        truncation=True
    )

    return {
        'input_ids': input_encodings['input_ids'],
        'attention_mask': input_encodings['attention_mask'],
        'labels': target_encodings['input_ids']
    }

In [8]:
dataset_samsum_pt = dataset_samsum.map(convert_examples_to_features, batched = True)

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

In [9]:
dataset_samsum_pt['train']

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14732
})

In [10]:
# Training

from transformers import DataCollatorForSeq2Seq
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [11]:
import accelerate
print(accelerate.__version__)

1.13.0


In [12]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir = 'pegasus-samsum', num_train_epochs= 5, warmup_steps=500,
    per_device_train_batch_size=1, per_device_eval_batch_size= 1,
    weight_decay=0.01, logging_steps= 10,
    eval_strategy="steps", eval_steps=500, save_steps=1e6,
    gradient_accumulation_steps=16
)

In [13]:
# Take only 25% of training data
train_dataset_small = dataset_samsum_pt["train"].shuffle(seed=42).select(
    range(int(0.25 * len(dataset_samsum_pt["train"])))
)

trainer = Trainer(
    model=model_pegasus,
    args=trainer_args,
    data_collator=seq2seq_data_collator,
    train_dataset=train_dataset_small,
    eval_dataset=dataset_samsum_pt["validation"]
)

In [14]:
trainer.train()

Step,Training Loss,Validation Loss
500,25.870511,1.502024
1000,21.932019,1.451666
1155,22.841972,1.450047


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1155, training_loss=27.082942589433678, metrics={'train_runtime': 3459.9763, 'train_samples_per_second': 5.322, 'train_steps_per_second': 0.334, 'total_flos': 6969802963230720.0, 'train_loss': 27.082942589433678, 'epoch': 5.0})

In [15]:
def generate_batch_sized_chunk(list_of_elements, batch_size):
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i+batch_size]


def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                                batch_size=16, device= device,
                                column_text= "article",
                                column_summary= "highlights"):
    article_batches = list(generate_batch_sized_chunk(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunk(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(zip(article_batches,target_batches), total= len(article_batches)):
        inputs = tokenizer(article_batch, max_length = 1024, truncation = True, padding = "max_length", return_tensors="pt")

        summaries = model.generate(input_ids= inputs["input_ids"].to(device),
                                   attention_mask= inputs["attention_mask"].to(device),
                                   length_penalty= 0.8, num_beams= 8, max_length=128)
        decoded_summaries = [tokenizer.decode(s, skip_special_tokens = True,
                                              clean_up_tokenization_spaces = True)
                                              for s in summaries]

        decoded_summaries = [d.replace("", " ") for d in decoded_summaries]

        metric.add_batch(predictions = decoded_summaries, references = target_batch)

    score = metric.compute()
    return score

In [3]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=0891aedcb36b0f97760c309ade3adb1d9b0d31af60782c84f7b226ac33a524c4
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [1]:
import evaluate
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_metric = evaluate.load("rouge")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
